# 17. Sequence recurrence and state-space updates — full-depth Mamba-130M topology at reduced tensor width

This notebook preserves the **Mamba-130M architecture depth and mixer topology** while reducing only tensor widths, vocabulary size, batch size, and sequence length for CPU execution.

Kept from the released Mamba-130M configuration and mixer:

- 24 Mamba blocks,
- RMSNorm pre-norm residual stack and final RMSNorm,
- tied token embedding / language-model head,
- expansion factor 2,
- causal depthwise convolution with kernel size 4,
- low-rank timestep path `x_proj -> dt_rank -> dt_proj`,
- input-dependent `B_t` and `C_t`, learned diagonal `A`, skip `D`,
- selective scan followed by the `SiLU(z)` gate and output projection.

Only dimensions such as `d_model`, state width, `dt_rank`, vocabulary size, batch size, and sequence length are reduced.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cpu")
print("device:", device)

## 1. Selective state-space scan

For each token, the mixer constructs a positive timestep `Delta_t` and input-dependent `B_t, C_t`. The diagonal continuous state matrix is discretized with `exp(Delta_t A)`, then the state is updated recurrently.

In [ ]:
def selective_scan_reference(u, delta, A, B, C, D):
    batch_size, channels, sequence_length = u.shape
    state_dim = A.size(1)

    state = torch.zeros(
        batch_size,
        channels,
        state_dim,
        device=u.device,
        dtype=u.dtype,
    )
    outputs = []

    for time_index in range(sequence_length):
        u_t = u[:, :, time_index]
        delta_t = delta[:, :, time_index]
        B_t = B[:, time_index, :]
        C_t = C[:, time_index, :]

        discrete_A = torch.exp(
            delta_t[:, :, None] * A[None, :, :]
        )
        discrete_B_times_input = (
            delta_t[:, :, None]
            * B_t[:, None, :]
            * u_t[:, :, None]
        )
        state = discrete_A * state + discrete_B_times_input

        y_t = torch.sum(
            state * C_t[:, None, :],
            dim=-1,
        )
        y_t = y_t + D[None, :] * u_t
        outputs.append(y_t)

    return torch.stack(outputs, dim=-1)

## 2. One Mamba mixer with the released parameter topology

The important point is that `Delta` is **not** produced by a direct full-width projection. The mixer first emits a low-rank timestep representation together with `B` and `C`, then expands the timestep through `dt_proj`.

In [ ]:
class MambaMixer(nn.Module):
    def __init__(
        self,
        model_dim=8,
        state_dim=4,
        expand=2,
        conv_kernel=4,
        dt_rank=2,
        dt_min=0.001,
        dt_max=0.1,
    ):
        super().__init__()
        self.model_dim = model_dim
        self.state_dim = state_dim
        self.inner_dim = expand * model_dim
        self.conv_kernel = conv_kernel
        self.dt_rank = dt_rank

        self.in_proj = nn.Linear(
            model_dim,
            2 * self.inner_dim,
            bias=False,
        )
        self.conv1d = nn.Conv1d(
            self.inner_dim,
            self.inner_dim,
            kernel_size=conv_kernel,
            groups=self.inner_dim,
            bias=True,
        )
        self.x_proj = nn.Linear(
            self.inner_dim,
            dt_rank + 2 * state_dim,
            bias=False,
        )
        self.dt_proj = nn.Linear(
            dt_rank,
            self.inner_dim,
            bias=True,
        )

        dt_init_std = dt_rank ** -0.5
        nn.init.uniform_(
            self.dt_proj.weight,
            -dt_init_std,
            dt_init_std,
        )

        initial_dt = torch.exp(
            torch.rand(self.inner_dim)
            * (math.log(dt_max) - math.log(dt_min))
            + math.log(dt_min)
        )
        inverse_softplus = initial_dt + torch.log(
            -torch.expm1(-initial_dt)
        )
        with torch.no_grad():
            self.dt_proj.bias.copy_(inverse_softplus)

        initial_A = torch.arange(
            1,
            state_dim + 1,
            dtype=torch.float32,
        ).repeat(self.inner_dim, 1)
        self.A_log = nn.Parameter(torch.log(initial_A))
        self.D = nn.Parameter(torch.ones(self.inner_dim))
        self.out_proj = nn.Linear(
            self.inner_dim,
            model_dim,
            bias=False,
        )

    def forward(self, hidden):
        projected = self.in_proj(hidden)
        x_branch, z_branch = projected.chunk(2, dim=-1)

        x_channels = x_branch.transpose(1, 2)
        x_channels = F.pad(
            x_channels,
            (self.conv_kernel - 1, 0),
        )
        convolved = F.silu(self.conv1d(x_channels))
        sequence_features = convolved.transpose(1, 2)

        ssm_parameters = self.x_proj(sequence_features)
        dt_low_rank, B, C = torch.split(
            ssm_parameters,
            [self.dt_rank, self.state_dim, self.state_dim],
            dim=-1,
        )
        delta = F.softplus(
            self.dt_proj(dt_low_rank)
        ).transpose(1, 2)
        A = -torch.exp(self.A_log.float())

        scanned = selective_scan_reference(
            convolved,
            delta,
            A,
            B,
            C,
            self.D,
        )
        gated = (
            scanned.transpose(1, 2)
            * F.silu(z_branch)
        )
        return self.out_proj(gated)

## 3. Full 24-block Mamba-130M stack

The production model uses width 768; that width is reduced here. The **24-block depth is not reduced**.

In [ ]:
class MambaResidualBlock(nn.Module):
    def __init__(self, model_dim=8):
        super().__init__()
        self.norm = nn.RMSNorm(model_dim, eps=1e-5)
        self.mixer = MambaMixer(model_dim=model_dim)

    def forward(self, hidden):
        return hidden + self.mixer(self.norm(hidden))


class SmallWidthMamba130M(nn.Module):
    def __init__(
        self,
        vocab_size=64,
        model_dim=8,
        num_layers=24,
    ):
        super().__init__()
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, model_dim)
        self.layers = nn.ModuleList(
            [
                MambaResidualBlock(model_dim=model_dim)
                for _ in range(num_layers)
            ]
        )
        self.final_norm = nn.RMSNorm(model_dim, eps=1e-5)

    def forward(self, token_ids):
        hidden = self.embedding(token_ids)
        for layer in self.layers:
            hidden = layer(hidden)
        hidden = self.final_norm(hidden)
        return F.linear(hidden, self.embedding.weight)


model = SmallWidthMamba130M().to(device)
tokens = torch.randint(0, 64, (1, 6), device=device)
target = torch.randint(0, 64, (1, 6), device=device)

logits = model(tokens)
loss = F.cross_entropy(
    logits.reshape(-1, logits.size(-1)),
    target.reshape(-1),
)
loss.backward()

print("logits:", logits.shape)
print("loss:", loss.item())

## 4. Structural and causality checks

The checks below make the architectural invariants explicit rather than relying on names or comments.

In [ ]:
assert len(model.layers) == 24
assert all(layer.mixer.conv_kernel == 4 for layer in model.layers)
assert all(
    layer.mixer.conv1d.groups == layer.mixer.inner_dim
    for layer in model.layers
)
assert all(
    layer.mixer.x_proj.out_features
    == layer.mixer.dt_rank + 2 * layer.mixer.state_dim
    for layer in model.layers
)
assert model.embedding.weight.grad is not None

with torch.no_grad():
    prefix_tokens = torch.randint(0, 64, (1, 6), device=device)
    changed_tokens = prefix_tokens.clone()
    changed_tokens[:, 4:] = torch.randint(0, 64, (1, 2), device=device)

    original_logits = model(prefix_tokens)
    changed_logits = model(changed_tokens)
    prefix_difference = (
        original_logits[:, :4] - changed_logits[:, :4]
    ).abs().max()

assert prefix_difference.item() < 1e-6
print("24 layers: PASS")
print("causal conv kernel 4: PASS")
print("low-rank dt path: PASS")
print("causality max prefix difference:", prefix_difference.item())

## References and provenance

- **Mamba / state-spaces/mamba**: selective SSM mixer, causal depthwise convolution, low-rank timestep parameterization, input-dependent `B/C`, gating, RMSNorm residual stack.
- **state-spaces/mamba-130m released config**: 24 layers, model dimension 768, RMSNorm, residual in FP32, tied embedding lineage. The implementation above keeps the 24-layer topology and reduces only tensor widths and execution-scale inputs.